# Capstone — Part 1: Data Acquisition, Cleaning, and Exploratory Analysis

**Dataset:** `housing_data.csv` — Synthetic Residential Housing Dataset  
**Author:** Student  
**Date:** 2026-06-30  

In [ ]:
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline

print('Libraries loaded successfully.')

---
## Task 0 — Generate Raw Dataset

In [ ]:
if not os.path.exists('housing_data.csv'):
    exec(open('generate_dataset.py').read())
    print('Dataset generated.')
else:
    print('housing_data.csv already exists — skipping generation.')

---
## Task 1 — Load & Initial Inspection

In [ ]:
df = pd.read_csv('housing_data.csv')

print('── First 5 rows ──')
display(df.head())

print('\n── Column dtypes ──')
print(df.dtypes)

print(f'\n── Shape: {df.shape[0]} rows × {df.shape[1]} columns ──')

---
## Task 2 — Null Value Analysis

In [ ]:
null_count = df.isnull().sum()
null_pct   = (df.isnull().sum() / df.shape[0]) * 100

null_report = pd.DataFrame({'null_count': null_count, 'null_pct (%)': null_pct})
null_report = null_report.sort_values('null_pct (%)', ascending=False)

print('Null count & percentage per column:')
display(null_report.style.format({'null_pct (%)': '{:.2f}'})
        .background_gradient(subset=['null_pct (%)'], cmap='Reds'))

HIGH_NULL_THRESHOLD = 20.0
high_null_cols = null_report[null_report['null_pct (%)'] > HIGH_NULL_THRESHOLD].index.tolist()
low_null_cols  = null_report[
    (null_report['null_pct (%)'] > 0) & (null_report['null_pct (%)'] <= HIGH_NULL_THRESHOLD)
].index.tolist()

print(f'\nColumns EXCEEDING {HIGH_NULL_THRESHOLD}% null rate → will NOT impute:')
print(high_null_cols)

print(f'\nColumns below {HIGH_NULL_THRESHOLD}% null rate → impute numeric with median:')
print(low_null_cols)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in low_null_cols:
    if col in numeric_cols:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled '{col}' with median = {median_val:.2f}")

---
## Task 3 — Duplicate Detection & Removal

In [ ]:
n_before  = df.shape[0]
n_dups    = df.duplicated().sum()
print(f'Duplicate rows found: {n_dups}')

null_pct_before = (df.isnull().sum() / df.shape[0] * 100).round(4)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
n_after = df.shape[0]

null_pct_after = (df.isnull().sum() / df.shape[0] * 100).round(4)
print(f'Rows before: {n_before}  →  after: {n_after}  (removed {n_before - n_after})')

diff = (null_pct_after - null_pct_before).round(4)
changed = diff[diff != 0]
print('\nChange in null % after deduplication:')
print(changed if len(changed) > 0 else '  No meaningful change in null percentages.')

---
## Task 4 — Data Type Correction

In [ ]:
mem_before = df.memory_usage(deep=True).sum()
print(f'Memory BEFORE: {mem_before:,} bytes')

# Fix dtype bug: OverallCond object → int
df['OverallCond'] = pd.to_numeric(df['OverallCond'], errors='coerce')
df['OverallCond'] = df['OverallCond'].fillna(df['OverallCond'].median()).astype(int)

# Ensure previously imputed float columns are clean
df['TotalBsmtSF'] = pd.to_numeric(df['TotalBsmtSF'], errors='coerce')
df['GarageCars']  = pd.to_numeric(df['GarageCars'],  errors='coerce')

# Convert repetitive string columns to category
cat_cols = ['Neighborhood', 'BldgType', 'HouseStyle', 'SaleCondition']
for col in cat_cols:
    df[col] = df[col].astype('category')

mem_after = df.memory_usage(deep=True).sum()
print(f'Memory AFTER:  {mem_after:,} bytes')
print(f'Memory saved:  {mem_before - mem_after:,} bytes  ({(1 - mem_after/mem_before)*100:.1f}% reduction)')

print('\nUpdated dtypes:')
print(df.dtypes)

---
## Task 5 — Descriptive Statistics & Skewness

In [ ]:
num_df = df.select_dtypes(include=[np.number]).drop(columns=['Id'], errors='ignore')

print('── df.describe() ──')
display(num_df.describe().round(2))

skewness = num_df.apply(lambda c: c.skew()).sort_values(key=abs, ascending=False)
print('\n── Skewness per numeric column ──')
display(skewness.to_frame('skewness').style
        .format('{:.4f}')
        .background_gradient(cmap='RdBu_r', vmin=-5, vmax=5))

most_skewed     = skewness.abs().idxmax()
most_skewed_val = skewness[most_skewed]
print(f"\nColumn with HIGHEST absolute skewness: '{most_skewed}'  (skew = {most_skewed_val:.4f})")

---
## Task 6 — Outlier Detection with IQR

In [ ]:
iqr_cols = ['SalePrice', 'LotArea']
records  = []

for col in iqr_cols:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lb  = q1 - 1.5 * iqr
    ub  = q3 + 1.5 * iqr
    n_out = int(((df[col] < lb) | (df[col] > ub)).sum())
    records.append({'Column': col, 'Q1': round(q1, 2), 'Q3': round(q3, 2),
                    'IQR': round(iqr, 2), 'Lower': round(lb, 2), 'Upper': round(ub, 2),
                    'Outliers': n_out, 'Outlier %': round(n_out / df.shape[0] * 100, 2)})

display(pd.DataFrame(records).set_index('Column'))
print('\nDecision: Outliers RETAINED — see README for justification.')

---
## Task 7 — Five Visualisations

In [ ]:
# 7a — Line plot
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df.sort_index()
ax.plot(plot_df.index, plot_df['SalePrice'], color='#4C72B0', linewidth=0.7, alpha=0.8)
ax.set_title('Sale Price by Row Index', fontsize=14, fontweight='bold')
ax.set_xlabel('Row Index')
ax.set_ylabel('Sale Price (USD)')
plt.tight_layout()
plt.savefig('figures/01_line_plot.png', dpi=150)
plt.show()

In [ ]:
# 7b — Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
mean_by_neigh = df.groupby('Neighborhood', observed=True)['SalePrice'].mean().sort_values(ascending=False)
colors = sns.color_palette('muted', len(mean_by_neigh))
ax.bar(mean_by_neigh.index, mean_by_neigh.values, color=colors, edgecolor='white')
ax.set_title('Mean Sale Price by Neighborhood', fontsize=14, fontweight='bold')
ax.set_xlabel('Neighborhood')
ax.set_ylabel('Mean Sale Price (USD)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('figures/02_bar_chart.png', dpi=150)
plt.show()

In [ ]:
# 7c — Histogram of most-skewed column
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df[most_skewed].dropna(), bins=20, kde=True,
             color='#C44E52', edgecolor='white', linewidth=0.5, ax=ax)
ax.set_title(f"Distribution of '{most_skewed}'  (skew = {most_skewed_val:.2f})",
             fontsize=14, fontweight='bold')
ax.set_xlabel(most_skewed)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('figures/03_histogram.png', dpi=150)
plt.show()

In [ ]:
# 7d — Scatter plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='GrLivArea', y='SalePrice',
                hue='OverallQual', palette='viridis',
                alpha=0.5, linewidth=0, ax=ax)
ax.set_title('Above-Grade Living Area vs. Sale Price\n(coloured by Overall Quality)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Above-Grade Living Area (sq ft)')
ax.set_ylabel('Sale Price (USD)')
plt.tight_layout()
plt.savefig('figures/04_scatter_plot.png', dpi=150)
plt.show()

In [ ]:
# 7e — Box plot
fig, ax = plt.subplots(figsize=(10, 6))
order = df.groupby('BldgType', observed=True)['SalePrice'].median() \
          .sort_values(ascending=False).index
sns.boxplot(data=df, x='BldgType', y='SalePrice',
            order=order, palette='Set2', ax=ax)
ax.set_title('Sale Price Distribution by Building Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Building Type')
ax.set_ylabel('Sale Price (USD)')
plt.tight_layout()
plt.savefig('figures/05_box_plot.png', dpi=150)
plt.show()

---
## Task 8 — Pearson Correlation Heat Map

In [ ]:
pearson_corr = num_df.corr(method='pearson')
print('Pearson correlation matrix:')
display(pearson_corr.round(3))

mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation Heat Map — Numeric Features',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('figures/06_correlation_heatmap.png', dpi=150)
plt.show()

corr_upper = pearson_corr.where(~mask)
top_corr   = corr_upper.stack().abs().idxmax()
top_val    = corr_upper.loc[top_corr]
print(f"\nHighest absolute Pearson correlation: {top_corr[0]} ↔ {top_corr[1]}  (r = {top_val:.3f})")

---
## Task 9a — Imputation Strategy Comparison (Mean vs Median)

In [ ]:
top2_skewed = skewness.abs().nlargest(2).index.tolist()
print(f'Two most-skewed columns: {top2_skewed}\n')

comparison = []
for col in top2_skewed:
    col_mean   = df[col].mean()
    col_median = df[col].median()
    col_skew   = df[col].skew()
    comparison.append({'Column': col, 'Skewness': round(col_skew, 4),
                       'Mean': round(col_mean, 4), 'Median': round(col_median, 4),
                       'Chosen Statistic': 'Median (skew > 0)'})
    nulls_remaining = df[col].isnull().sum()
    if nulls_remaining > 0:
        df[col].fillna(col_median, inplace=True)

display(pd.DataFrame(comparison).set_index('Column'))

print('\nFinal isnull().sum():')
print(df.isnull().sum())

---
## Task 9b — Spearman Rank Correlation

In [ ]:
spearman_corr = num_df.corr(method='spearman')
diff_matrix   = (spearman_corr - pearson_corr).abs()

mask_upper  = np.triu(np.ones_like(diff_matrix, dtype=bool), k=1)
diff_upper  = diff_matrix.where(mask_upper)
top3_diff   = diff_upper.stack().nlargest(3)

print('Pearson matrix:')
display(pearson_corr.round(3))

print('\nSpearman matrix:')
display(spearman_corr.round(3))

print('\nTop-3 pairs by |Spearman − Pearson|:')
display(top3_diff.reset_index().rename(
    columns={'level_0': 'Column A', 'level_1': 'Column B', 0: '|Spearman − Pearson|'}
))

# Three-panel heatmap
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, (matrix, title) in zip(
    axes,
    [(pearson_corr, 'Pearson'), (spearman_corr, 'Spearman'),
     (diff_matrix,  '|Spearman − Pearson|')]
):
    cmap   = 'coolwarm' if 'Pearson' in title or 'Spearman' in title else 'YlOrRd'
    center = 0          if 'Pearson' in title or 'Spearman' in title else None
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap=cmap,
                center=center, linewidths=0.3, linecolor='white',
                ax=ax, cbar_kws={'shrink': 0.7})
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Pearson vs. Spearman Correlation Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/07_spearman_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 9c — Grouped Aggregation

In [ ]:
cat_col = 'Neighborhood'
num_col = 'SalePrice'

grouped = df.groupby(cat_col, observed=True)[num_col].agg(['mean', 'std', 'count'])
grouped = grouped.sort_values('mean', ascending=False)

print(f'Grouped aggregation: {cat_col} → {num_col}\n')
display(grouped.style.format({'mean': '${:,.0f}', 'std': '${:,.0f}', 'count': '{:.0f}'})
        .background_gradient(subset=['mean'], cmap='Greens'))

highest_mean = grouped['mean'].idxmax()
highest_std  = grouped['std'].idxmax()
mean_ratio   = grouped['mean'].max() / grouped['mean'].min()

print(f'\nHighest-mean group : {highest_mean}  (mean = ${grouped.loc[highest_mean, "mean"]:,.0f})')
print(f'Highest-std  group : {highest_std}   (std  = ${grouped.loc[highest_std,  "std"]:,.0f})')
print(f'Mean ratio (max/min): {mean_ratio:.2f}')

---
## Task 10 — Save Cleaned Dataset

In [ ]:
df.to_csv('cleaned_data.csv', index=False)
print(f'cleaned_data.csv saved: {df.shape[0]} rows × {df.shape[1]} columns')
print('\n✓ Part 1 complete — all outputs written to disk.')